In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

from aymurai.models.sentence_encoder.base import BaseSentenceEncoder
from aymurai.transforms.entity_subcategories.sentence_transformer import (
    SentenceTransformerSubcategorizer,
)


class FakeEncoder(BaseSentenceEncoder):
    def _vec(self, text: str) -> np.ndarray:
        # Deterministic 8-dim vector from hash
        h = abs(hash(text))
        return np.array([(h >> i) & 0xFF for i in range(0, 64, 8)], dtype=np.float32)

    def encode(self, text_array, encoder_type, context_array=None):
        return self.batch_encode(text_array, encoder_type)

    def batch_encode(self, text_array, encoder_type, batch_size: int = 256):
        texts = list(text_array)
        return np.vstack([self._vec(self.normalize_text(t)) for t in texts])


tmpdir = Path(tempfile.mkdtemp())
subcats_path = tmpdir / "subcats.txt"
subcats_path.write_text("\n".join(["alpha", "beta", "gamma"]))

# BM25 + encoder path
hybrid_embeddings = tmpdir / "embeddings_hybrid.npz"
hybrid = SentenceTransformerSubcategorizer(
    category="CONDUCTA",
    embeddings_path=str(hybrid_embeddings),
    encoder_name="distiluse",
    bm25_weight=0.5,
    encoder=FakeEncoder(),
)
print(hybrid.batch_retrieve(["alpha case", "beta story"], top_k=2))

# Encoder-only path (bm25_weight = 0)
encoder_only_embeddings = tmpdir / "embeddings_encoder_only.npz"
encoder_only = SentenceTransformerSubcategorizer(
    category="CONDUCTA",
    embeddings_path=str(encoder_only_embeddings),
    encoder_name="distiluse",
    bm25_weight=0.0,
    encoder=FakeEncoder(),
    rebuild_embeddings=True,
)
print(encoder_only.batch_retrieve(["gamma topic"], top_k=2))